# Module 6: Large Language Models

This notebook covers the core components of Large Language Models (LLMs).

**Topics covered:**
- Tokenization (BPE)
- Causal language modeling
- Sampling strategies
- LoRA fine-tuning
- RLHF concepts

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
import re

np.random.seed(42)
%matplotlib inline

## 6.1 Byte Pair Encoding (BPE) Tokenization

In [ ]:
class BPETokenizer:
    """
    Simple BPE tokenizer implementation.
    
    BPE iteratively merges the most frequent adjacent pairs of tokens.
    """
    
    def __init__(self, vocab_size=256):
        self.vocab_size = vocab_size
        self.merges = {}  # (pair) -> new_token
        self.vocab = {}   # token -> id
        self.inverse_vocab = {}  # id -> token
    
    def get_pairs(self, word):
        """Get all adjacent pairs in a word."""
        pairs = []
        for i in range(len(word) - 1):
            pairs.append((word[i], word[i + 1]))
        return pairs
    
    def train(self, texts):
        """Train BPE on a corpus."""
        # Initialize with character vocabulary
        word_freqs = Counter()
        for text in texts:
            words = text.split()
            for word in words:
                # Add end-of-word marker
                word = tuple(word) + ('</w>',)
                word_freqs[word] += 1
        
        # Build initial character vocab
        chars = set()
        for word in word_freqs:
            chars.update(word)
        
        for i, char in enumerate(sorted(chars)):
            self.vocab[char] = i
            self.inverse_vocab[i] = char
        
        # Iteratively merge most frequent pairs
        num_merges = self.vocab_size - len(self.vocab)
        
        for merge_idx in range(num_merges):
            # Count pair frequencies
            pair_freqs = Counter()
            for word, freq in word_freqs.items():
                pairs = self.get_pairs(word)
                for pair in pairs:
                    pair_freqs[pair] += freq
            
            if not pair_freqs:
                break
            
            # Find most frequent pair
            best_pair = pair_freqs.most_common(1)[0][0]
            
            # Create new token
            new_token = best_pair[0] + best_pair[1]
            if isinstance(best_pair[0], str) and isinstance(best_pair[1], str):
                new_token = best_pair[0] + best_pair[1]
            
            # Add to vocabulary
            new_id = len(self.vocab)
            self.vocab[new_token] = new_id
            self.inverse_vocab[new_id] = new_token
            self.merges[best_pair] = new_token
            
            # Merge in all words
            new_word_freqs = Counter()
            for word, freq in word_freqs.items():
                new_word = []
                i = 0
                while i < len(word):
                    if i < len(word) - 1 and (word[i], word[i+1]) == best_pair:
                        new_word.append(new_token)
                        i += 2
                    else:
                        new_word.append(word[i])
                        i += 1
                new_word_freqs[tuple(new_word)] = freq
            
            word_freqs = new_word_freqs
            
            if merge_idx < 10:
                print(f"Merge {merge_idx + 1}: {best_pair} -> {new_token}")
        
        print(f"\nFinal vocabulary size: {len(self.vocab)}")
    
    def encode(self, text):
        """Encode text to token ids."""
        words = text.split()
        tokens = []
        
        for word in words:
            word = list(word) + ['</w>']
            
            # Apply merges
            while len(word) > 1:
                pairs = self.get_pairs(word)
                mergeable = [(p, self.merges[p]) for p in pairs if p in self.merges]
                
                if not mergeable:
                    break
                
                # Apply first applicable merge
                best_pair, new_token = mergeable[0]
                new_word = []
                i = 0
                while i < len(word):
                    if i < len(word) - 1 and (word[i], word[i+1]) == best_pair:
                        new_word.append(new_token)
                        i += 2
                    else:
                        new_word.append(word[i])
                        i += 1
                word = new_word
            
            # Convert to ids
            for token in word:
                if token in self.vocab:
                    tokens.append(self.vocab[token])
        
        return tokens
    
    def decode(self, ids):
        """Decode token ids back to text."""
        tokens = [self.inverse_vocab[i] for i in ids if i in self.inverse_vocab]
        text = ''.join(tokens)
        text = text.replace('</w>', ' ')
        return text.strip()

In [ ]:
# Train BPE on a small corpus
corpus = [
    "low lower lowest",
    "new newer newest",
    "the cat sat on the mat",
    "the quick brown fox jumped over the lazy dog",
    "deep learning is learning with deep networks",
    "neural networks learn representations"
]

tokenizer = BPETokenizer(vocab_size=50)
tokenizer.train(corpus)

In [ ]:
# Test tokenization
test_texts = ["the cat", "newer", "deep learning"]

for text in test_texts:
    ids = tokenizer.encode(text)
    decoded = tokenizer.decode(ids)
    print(f"'{text}' -> {ids} -> '{decoded}'")

## 6.2 Causal Language Modeling

In [ ]:
def softmax(x, axis=-1):
    """Numerically stable softmax."""
    x = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

def cross_entropy_loss(logits, targets):
    """Cross-entropy loss for language modeling."""
    probs = softmax(logits)
    # Select probability of correct token
    log_probs = np.log(probs[np.arange(len(targets)), targets] + 1e-10)
    return -np.mean(log_probs)

class SimpleLM:
    """
    Simple causal language model (for demonstration).
    
    In practice, this would be a Transformer, but we use a simple
    embedding + linear layer for clarity.
    """
    
    def __init__(self, vocab_size, embed_dim, context_size):
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.context_size = context_size
        
        # Token embeddings
        scale = np.sqrt(2.0 / embed_dim)
        self.embeddings = np.random.randn(vocab_size, embed_dim) * scale
        
        # Simple linear prediction (in practice: Transformer)
        self.W_out = np.random.randn(embed_dim, vocab_size) * scale
    
    def forward(self, input_ids):
        """
        Forward pass: input_ids -> logits
        
        Args:
            input_ids: (seq_len,) array of token ids
        Returns:
            logits: (seq_len, vocab_size) array
        """
        # Embed tokens
        embeds = self.embeddings[input_ids]  # (seq_len, embed_dim)
        
        # Predict next token logits
        logits = embeds @ self.W_out  # (seq_len, vocab_size)
        
        return logits
    
    def compute_loss(self, input_ids):
        """
        Compute language modeling loss.
        
        For causal LM: predict next token at each position.
        """
        # Shift: input[:-1] predicts input[1:]
        inputs = input_ids[:-1]
        targets = input_ids[1:]
        
        logits = self.forward(inputs)
        loss = cross_entropy_loss(logits, targets)
        
        return loss

In [ ]:
# Demonstrate causal LM training objective
vocab_size = 50
embed_dim = 32
context_size = 8

lm = SimpleLM(vocab_size, embed_dim, context_size)

# Fake input sequence
input_ids = np.array([1, 5, 3, 8, 2, 9, 4, 7])

loss = lm.compute_loss(input_ids)
print(f"Loss: {loss:.4f}")
print(f"Perplexity: {np.exp(loss):.2f}")

# Show the prediction task
print("\nCausal LM predicts:")
for i in range(len(input_ids) - 1):
    print(f"  Token {input_ids[i]} -> predict {input_ids[i+1]}")

## 6.3 Sampling Strategies

In [ ]:
def greedy_decode(logits):
    """Greedy decoding: always pick highest probability."""
    return np.argmax(logits)

def temperature_sample(logits, temperature=1.0):
    """Sample with temperature."""
    if temperature == 0:
        return greedy_decode(logits)
    
    scaled_logits = logits / temperature
    probs = softmax(scaled_logits)
    return np.random.choice(len(probs), p=probs)

def top_k_sample(logits, k=10, temperature=1.0):
    """Top-k sampling: only consider top k tokens."""
    # Get top k indices
    top_k_idx = np.argsort(logits)[-k:]
    
    # Mask out other tokens
    masked_logits = np.full_like(logits, -np.inf)
    masked_logits[top_k_idx] = logits[top_k_idx]
    
    return temperature_sample(masked_logits, temperature)

def top_p_sample(logits, p=0.9, temperature=1.0):
    """Top-p (nucleus) sampling: sample from smallest set with cumprob >= p."""
    probs = softmax(logits / temperature)
    sorted_idx = np.argsort(probs)[::-1]  # descending
    sorted_probs = probs[sorted_idx]
    
    # Find cutoff
    cumsum = np.cumsum(sorted_probs)
    cutoff_idx = np.searchsorted(cumsum, p) + 1
    
    # Keep only top-p tokens
    top_p_idx = sorted_idx[:cutoff_idx]
    top_p_probs = probs[top_p_idx]
    top_p_probs = top_p_probs / top_p_probs.sum()  # renormalize
    
    return np.random.choice(top_p_idx, p=top_p_probs)

In [ ]:
# Visualize sampling strategies
np.random.seed(42)

# Create example logits with a peaked distribution
vocab_size = 20
logits = np.random.randn(vocab_size)
logits[5] = 3.0  # Make token 5 most likely
logits[8] = 2.0  # Token 8 second most likely

probs = softmax(logits)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

# Original distribution
axes[0, 0].bar(range(vocab_size), probs)
axes[0, 0].set_title('Original Probabilities')
axes[0, 0].set_xlabel('Token')
axes[0, 0].set_ylabel('Probability')

# Temperature = 0.5 (sharper)
axes[0, 1].bar(range(vocab_size), softmax(logits / 0.5))
axes[0, 1].set_title('Temperature = 0.5 (more focused)')
axes[0, 1].set_xlabel('Token')

# Temperature = 2.0 (flatter)
axes[0, 2].bar(range(vocab_size), softmax(logits / 2.0))
axes[0, 2].set_title('Temperature = 2.0 (more random)')
axes[0, 2].set_xlabel('Token')

# Sample distributions
n_samples = 1000

# Greedy
greedy_samples = [greedy_decode(logits) for _ in range(n_samples)]
axes[1, 0].hist(greedy_samples, bins=vocab_size, range=(0, vocab_size), density=True)
axes[1, 0].set_title('Greedy (always token 5)')
axes[1, 0].set_xlabel('Token')

# Top-k
topk_samples = [top_k_sample(logits, k=5) for _ in range(n_samples)]
axes[1, 1].hist(topk_samples, bins=vocab_size, range=(0, vocab_size), density=True)
axes[1, 1].set_title('Top-k (k=5)')
axes[1, 1].set_xlabel('Token')

# Top-p
topp_samples = [top_p_sample(logits, p=0.9) for _ in range(n_samples)]
axes[1, 2].hist(topp_samples, bins=vocab_size, range=(0, vocab_size), density=True)
axes[1, 2].set_title('Top-p (p=0.9)')
axes[1, 2].set_xlabel('Token')

plt.tight_layout()
plt.show()

## 6.4 LoRA (Low-Rank Adaptation)

In [ ]:
class LoRALayer:
    """
    LoRA: Low-Rank Adaptation of Large Language Models.
    
    Instead of fine-tuning all weights W, we learn:
    W' = W + B @ A
    
    Where B is (d_out, r) and A is (r, d_in) with r << min(d_in, d_out)
    """
    
    def __init__(self, original_weight, rank=4, alpha=1.0):
        """
        Args:
            original_weight: The frozen pre-trained weight (d_out, d_in)
            rank: The rank of the low-rank matrices
            alpha: Scaling factor for the LoRA update
        """
        self.W = original_weight  # Frozen
        self.rank = rank
        self.alpha = alpha
        
        d_out, d_in = original_weight.shape
        
        # Initialize LoRA matrices
        # A: Gaussian init, B: Zero init (so initial update is zero)
        self.A = np.random.randn(rank, d_in) * 0.01
        self.B = np.zeros((d_out, rank))
        
        # Scaling
        self.scaling = alpha / rank
    
    def forward(self, x):
        """Forward pass with LoRA."""
        # Original output
        out = x @ self.W.T
        
        # LoRA contribution
        lora_out = (x @ self.A.T @ self.B.T) * self.scaling
        
        return out + lora_out
    
    def num_trainable_params(self):
        """Count trainable parameters."""
        return self.A.size + self.B.size
    
    def num_original_params(self):
        """Count original parameters."""
        return self.W.size
    
    def merge_weights(self):
        """Merge LoRA weights into original for inference."""
        return self.W + self.scaling * (self.B @ self.A)

In [ ]:
# Demonstrate LoRA efficiency
d_in, d_out = 4096, 4096  # Typical LLM dimensions
ranks = [1, 4, 8, 16, 32, 64]

original_params = d_in * d_out
print(f"Original parameters: {original_params:,} ({original_params / 1e6:.1f}M)")
print("\nLoRA comparison:")
print("-" * 50)

for rank in ranks:
    lora_params = rank * d_in + rank * d_out
    percentage = 100 * lora_params / original_params
    print(f"Rank {rank:2d}: {lora_params:,} params ({percentage:.2f}% of original)")

In [ ]:
# Test LoRA layer
np.random.seed(42)

# Original weight matrix
W = np.random.randn(256, 256) * 0.01

# Create LoRA layer
lora = LoRALayer(W, rank=8, alpha=16)

# Test forward pass
x = np.random.randn(1, 256)

# Before training (B is zero, so output = original)
out_original = x @ W.T
out_lora = lora.forward(x)

print("Before training (B=0):")
print(f"  Difference: {np.abs(out_original - out_lora).max():.6f}")

# Simulate some training (random B update)
lora.B = np.random.randn(*lora.B.shape) * 0.1
out_lora_trained = lora.forward(x)

print("\nAfter training:")
print(f"  Difference from original: {np.abs(out_original - out_lora_trained).mean():.4f}")

# Merge for inference
W_merged = lora.merge_weights()
out_merged = x @ W_merged.T

print(f"\nMerged output matches LoRA: {np.allclose(out_lora_trained, out_merged)}")

## 6.5 Reward Model and RLHF Concepts

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

class SimpleRewardModel:
    """
    A simple reward model that scores responses.
    
    Trained on preference pairs (chosen > rejected).
    """
    
    def __init__(self, feature_dim):
        self.feature_dim = feature_dim
        # Simple linear reward head
        self.W = np.random.randn(feature_dim) * 0.01
        self.b = 0.0
    
    def score(self, features):
        """Compute reward score for a response."""
        return np.dot(features, self.W) + self.b
    
    def preference_loss(self, chosen_features, rejected_features):
        """
        Bradley-Terry preference loss.
        
        Loss = -log(sigmoid(r_chosen - r_rejected))
        
        We want P(chosen > rejected) to be high.
        """
        r_chosen = self.score(chosen_features)
        r_rejected = self.score(rejected_features)
        
        loss = -np.log(sigmoid(r_chosen - r_rejected) + 1e-10)
        return loss
    
    def train_step(self, chosen_features, rejected_features, lr=0.01):
        """One training step."""
        r_chosen = self.score(chosen_features)
        r_rejected = self.score(rejected_features)
        
        # Gradient of Bradley-Terry loss
        diff = r_chosen - r_rejected
        grad_coef = sigmoid(diff) - 1  # = -sigmoid(-diff)
        
        # Update weights
        grad_W = grad_coef * (chosen_features - rejected_features)
        grad_b = grad_coef
        
        self.W -= lr * grad_W
        self.b -= lr * grad_b
        
        return self.preference_loss(chosen_features, rejected_features)

In [ ]:
# Train a simple reward model on synthetic preferences
np.random.seed(42)

feature_dim = 10
rm = SimpleRewardModel(feature_dim)

# Create synthetic preference data
# "Good" responses have high values in first half of features
# "Bad" responses have high values in second half
n_pairs = 200

chosen_data = []
rejected_data = []

for _ in range(n_pairs):
    chosen = np.random.randn(feature_dim)
    chosen[:5] += 1.0  # Good responses are positive in first half
    
    rejected = np.random.randn(feature_dim)
    rejected[5:] += 1.0  # Bad responses are positive in second half
    
    chosen_data.append(chosen)
    rejected_data.append(rejected)

# Train
losses = []
for epoch in range(50):
    epoch_loss = 0
    for c, r in zip(chosen_data, rejected_data):
        loss = rm.train_step(c, r, lr=0.1)
        epoch_loss += loss
    losses.append(epoch_loss / n_pairs)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Reward Model Training')

plt.subplot(1, 2, 2)
plt.bar(range(feature_dim), rm.W)
plt.xlabel('Feature')
plt.ylabel('Weight')
plt.title('Learned Reward Weights\n(first 5 should be positive)')
plt.axhline(y=0, color='k', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Test the trained reward model
print("Testing reward model:")
print("-" * 40)

# Good response (positive in first half)
good = np.zeros(feature_dim)
good[:5] = 1.0
print(f"Good response score: {rm.score(good):.4f}")

# Bad response (positive in second half)
bad = np.zeros(feature_dim)
bad[5:] = 1.0
print(f"Bad response score: {rm.score(bad):.4f}")

# Neutral response
neutral = np.zeros(feature_dim)
print(f"Neutral response score: {rm.score(neutral):.4f}")

# Preference accuracy
correct = 0
for c, r in zip(chosen_data, rejected_data):
    if rm.score(c) > rm.score(r):
        correct += 1
print(f"\nPreference accuracy: {100 * correct / n_pairs:.1f}%")

## 6.6 Perplexity and Evaluation

In [ ]:
def compute_perplexity(logits, targets):
    """
    Compute perplexity.
    
    Perplexity = exp(average cross-entropy loss)
    
    Lower is better. A perplexity of k means the model is as uncertain
    as if it had to choose uniformly among k options.
    """
    probs = softmax(logits)
    log_probs = np.log(probs[np.arange(len(targets)), targets] + 1e-10)
    avg_nll = -np.mean(log_probs)
    return np.exp(avg_nll)

# Demonstrate perplexity
vocab_size = 100
seq_len = 50

# Random model (uniform distribution)
random_logits = np.zeros((seq_len, vocab_size))  # Uniform
targets = np.random.randint(0, vocab_size, seq_len)
ppl_random = compute_perplexity(random_logits, targets)
print(f"Random model perplexity: {ppl_random:.2f}")
print(f"  (Expected: ~{vocab_size} for uniform distribution)")

# Slightly trained model (peaked distribution)
trained_logits = np.random.randn(seq_len, vocab_size) * 0.5
trained_logits[np.arange(seq_len), targets] += 2.0  # Boost correct tokens
ppl_trained = compute_perplexity(trained_logits, targets)
print(f"\nTrained model perplexity: {ppl_trained:.2f}")

# Well-trained model (very peaked)
good_logits = np.zeros((seq_len, vocab_size))
good_logits[np.arange(seq_len), targets] = 5.0
ppl_good = compute_perplexity(good_logits, targets)
print(f"Well-trained model perplexity: {ppl_good:.2f}")

## Summary

In this notebook, we covered:

1. **BPE Tokenization**: Building vocabulary by iteratively merging frequent pairs
2. **Causal Language Modeling**: Predicting next tokens from context
3. **Sampling Strategies**: Temperature, top-k, top-p for controlling randomness
4. **LoRA**: Efficient fine-tuning via low-rank weight updates
5. **Reward Modeling**: Training models on human preferences for RLHF
6. **Perplexity**: Evaluating language model quality

**Next:** Module 7 covers Generative Models (VAE, GAN, Diffusion).